# Forest Inspection - Training Notebook

Train semantic segmentation models.

**Models**: U-Net, U-Net++, DeepLabV3+

In [ ]:
# === ENVIRONMENT DETECTION & DATA CONFIG ===
import os, requests, zipfile
from pathlib import Path
from tqdm import tqdm

IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    DATA_ROOTS = [
        Path('/kaggle/input/datasets/leehien12/uav-forest-sunny-part1'),
        Path('/kaggle/input/datasets/khesng/uav-forest-sunny-next'),
    ]
else:
    DATA_ROOTS = [Path('data/forest_sunny')]

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZENODO_BASE = 'https://zenodo.org/records/15511426/files'

def download_seq(seq_num, target_dir):
    seq_dir = target_dir / f'seq{seq_num}'
    if seq_dir.exists() and len(list(seq_dir.rglob('*.png'))) > 0:
        print(f'[OK] seq{seq_num} da co san')
        return
    url = f'{ZENODO_BASE}/seq{seq_num}.zip?download=1'
    print(f'[DOWNLOAD] seq{seq_num}...')
    resp = requests.get(url, stream=True, timeout=60)
    total = int(resp.headers.get('content-length', 0))
    zip_path = target_dir / f'seq{seq_num}.zip'
    target_dir.mkdir(parents=True, exist_ok=True)
    with open(zip_path, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc=f'seq{seq_num}') as pbar:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    print(f'[EXTRACT] seq{seq_num}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)
    zip_path.unlink()
    print(f'[OK] seq{seq_num} done')

if not IS_KAGGLE:
    for seq in [1, 2, 3, 4]:
        download_seq(seq, DATA_ROOTS[0])

print(f'\n[DONE] DATA_ROOTS = {DATA_ROOTS}')

In [ ]:
# === INLINE SOURCE CODE ===
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from typing import Optional, Dict, List, Tuple, Callable, Union
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# --- Constants ---
LABEL_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65),
    (130, 76, 0), (152, 251, 152), (151, 126, 171), (250, 150, 0),
    (115, 176, 195), (123, 123, 123), (0, 0, 0),
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]
NUM_CLASSES = len(CLASS_NAMES)
SEQUENCE_INFO = {
    'seq1': {'pitch': 0, 'altitude': 30},
    'seq2': {'pitch': -60, 'altitude': 30},
    'seq3': {'pitch': -90, 'altitude': 30},
    'seq4': {'pitch': 0, 'altitude': 50},
    'seq5': {'pitch': -60, 'altitude': 50},
    'seq6': {'pitch': -90, 'altitude': 50},
    'seq7': {'pitch': 0, 'altitude': 80},
    'seq8': {'pitch': -60, 'altitude': 80},
    'seq9': {'pitch': -90, 'altitude': 80},
}

def rgb_to_class_id(label_rgb):
    h, w, _ = label_rgb.shape
    class_map = np.full((h, w), NUM_CLASSES - 1, dtype=np.int64)
    for cid, color in enumerate(LABEL_COLORS):
        mask = np.all(label_rgb == np.array(color, dtype=np.uint8), axis=-1)
        class_map[mask] = cid
    return class_map

def class_id_to_rgb(class_map):
    h, w = class_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, color in enumerate(LABEL_COLORS):
        rgb[class_map == cid] = color
    return rgb

# --- Dataset (multi-root + double-nesting support) ---
def _find_seq_dir(roots, seq):
    for root in roots:
        flat = root / seq
        if (flat / 'color').exists(): return flat
        nested = root / seq / seq
        if (nested / 'color').exists(): return nested
    return None

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        if isinstance(root, (str, Path)): self.roots = [Path(root)]
        else: self.roots = [Path(r) for r in root]
        self.transform = transform
        self.samples = []
        for seq in sequences:
            seq_dir = _find_seq_dir(self.roots, seq)
            if seq_dir is None:
                print(f'Warning: {seq} not found, skipping.')
                continue
            for cf in sorted((seq_dir / 'color').glob('*.png')):
                lf = seq_dir / 'labels' / cf.name
                if lf.exists():
                    self.samples.append((cf, lf))
        print(f'ForestDataset: {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        cp, lp = self.samples[idx]
        image = np.array(Image.open(cp).convert('RGB'))
        label_rgb = np.array(Image.open(lp).convert('RGB'))
        mask = rgb_to_class_id(label_rgb)
        if self.transform:
            t = self.transform(image=image, mask=mask)
            image, mask = t['image'], t['mask']
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).long()
        return image, mask.long()

# --- Transforms ---
def get_train_transforms(img_size=(512, 512)):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3, border_mode=0),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.1),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=(512, 512)):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# --- Splits ---
def get_available_sequences(data_roots):
    if isinstance(data_roots, (str, Path)): data_roots = [data_roots]
    found = set()
    for root in data_roots:
        root = Path(root)
        if not root.exists(): continue
        for d in root.iterdir():
            if not d.is_dir() or not d.name.startswith('seq'): continue
            if (d / 'color').exists() or (d / d.name / 'color').exists():
                found.add(d.name)
    return sorted(found)

def print_split_info(split):
    for subset, seqs in split.items():
        if not seqs: continue
        print(f'\n{subset.upper()}:')
        for seq in seqs:
            info = SEQUENCE_INFO.get(seq, {})
            print(f'  {seq}: pitch={info.get("pitch", "?")} deg, alt={info.get("altitude", "?")}m')

# --- Model ---
class UNet(nn.Module):
    def __init__(self, encoder='resnet34', encoder_weights='imagenet', num_classes=11, activation=None):
        super().__init__()
        self.model = smp.Unet(encoder_name=encoder, encoder_weights=encoder_weights,
                              in_channels=3, classes=num_classes, activation=activation)
        self.encoder_name = encoder
        self.num_classes = num_classes
    def forward(self, x): return self.model(x)

MODEL_REGISTRY = {
    'unet': lambda **kw: UNet(**kw),
    'unetpp': lambda encoder='resnet34', encoder_weights='imagenet', num_classes=11, **kw:
        smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
    'deeplabv3plus': lambda encoder='resnet50', encoder_weights='imagenet', num_classes=11, **kw:
        smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
}

def build_model(name, **kwargs):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown model: {name}. Available: {list(MODEL_REGISTRY.keys())}')
    return MODEL_REGISTRY[name](**kwargs)

# --- Losses ---
class DiceLoss(nn.Module):
    def __init__(self, num_classes=11, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        inter = (probs * targets_oh).sum(dim=dims)
        card = probs.sum(dim=dims) + targets_oh.sum(dim=dims)
        return 1.0 - ((2.0 * inter + self.smooth) / (card + self.smooth)).mean()

class CEDiceLoss(nn.Module):
    def __init__(self, num_classes=11, ce_weight=1.0, dice_weight=0.5, class_weights=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes=num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
    def forward(self, logits, targets):
        return self.ce_weight * self.ce(logits, targets) + self.dice_weight * self.dice(logits, targets)

# --- Metrics ---
class SegmentationMetrics:
    def __init__(self, num_classes, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [str(i) for i in range(num_classes)]
        self.confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)
    @torch.no_grad()
    def update(self, preds, targets):
        p, t = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        valid = (t >= 0) & (t < self.num_classes)
        p, t = p[valid], t[valid]
        cm = np.bincount(t * self.num_classes + p, minlength=self.num_classes**2)
        self.confusion_matrix += cm.reshape(self.num_classes, self.num_classes)
    def compute(self):
        cm = self.confusion_matrix
        inter = np.diag(cm)
        union = cm.sum(1) + cm.sum(0) - inter
        iou = np.where(union > 0, inter / union, 0.0)
        class_total = cm.sum(axis=1)
        class_acc = np.where(class_total > 0, inter / class_total, 0.0)
        pixel_acc = inter.sum() / cm.sum() if cm.sum() > 0 else 0.0
        valid = union > 0
        miou = iou[valid].mean() if valid.any() else 0.0
        result = {'miou': float(miou), 'pixel_accuracy': float(pixel_acc),
                  'mean_class_accuracy': float(class_acc[valid].mean()) if valid.any() else 0.0}
        for i, name in enumerate(self.class_names):
            result[f'iou_{name}'] = float(iou[i])
        return result
    def get_confusion_matrix(self):
        return self.confusion_matrix.copy()

# --- Trainer ---
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, config=None, device='cuda'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config = config or {}
        self.device = device
        self.use_amp = self.config.get('amp', True)
        self.scaler = GradScaler('cuda', enabled=self.use_amp)
        self.accumulation_steps = self.config.get('accumulation_steps', 1)
        self.writer = SummaryWriter(log_dir=self.config.get('log_dir', 'outputs/logs'))
        self.checkpoint_dir = Path(self.config.get('checkpoint_dir', 'outputs/checkpoints'))
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.save_top_k = self.config.get('save_top_k', 3)
        self.patience = self.config.get('early_stopping', {}).get('patience', 15)
        self.best_metric = 0.0
        self.epochs_without_improvement = 0
        self.metrics = SegmentationMetrics(self.config.get('num_classes', 11), CLASS_NAMES)
        self.global_step = 0
        self.best_checkpoints = []

    def train_epoch(self, epoch):
        self.model.train()
        total_loss, num_batches = 0.0, 0
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [Train]')
        self.optimizer.zero_grad()
        for batch_idx, (images, masks) in enumerate(pbar):
            images, masks = images.to(self.device), masks.to(self.device)
            with autocast(device_type='cuda', enabled=self.use_amp):
                loss = self.criterion(self.model(images), masks) / self.accumulation_steps
            self.scaler.scale(loss).backward()
            if (batch_idx + 1) % self.accumulation_steps == 0:
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
            total_loss += loss.item() * self.accumulation_steps
            num_batches += 1
            self.global_step += 1
            pbar.set_postfix({'loss': f'{loss.item() * self.accumulation_steps:.4f}'})
        return total_loss / max(num_batches, 1)

    @torch.no_grad()
    def validate(self, epoch):
        self.model.eval()
        self.metrics.reset()
        total_loss, num_batches = 0.0, 0
        for images, masks in tqdm(self.val_loader, desc=f'Epoch {epoch} [Val]'):
            images, masks = images.to(self.device), masks.to(self.device)
            with autocast(device_type='cuda', enabled=self.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
            total_loss += loss.item()
            num_batches += 1
            self.metrics.update(outputs.argmax(dim=1), masks)
        results = self.metrics.compute()
        results['val_loss'] = total_loss / max(num_batches, 1)
        return results

    def save_checkpoint(self, epoch, metric):
        path = self.checkpoint_dir / f'epoch_{epoch:03d}_miou_{metric:.4f}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'metric': metric}, path)
        self.best_checkpoints.append((metric, path))
        self.best_checkpoints.sort(key=lambda x: x[0])
        while len(self.best_checkpoints) > self.save_top_k:
            _, old = self.best_checkpoints.pop(0)
            if old.exists(): old.unlink()

    def fit(self, num_epochs):
        print(f'\nStarting training for {num_epochs} epochs on {self.device}\n')
        best_results = {}
        for epoch in range(1, num_epochs + 1):
            start = time.time()
            train_loss = self.train_epoch(epoch)
            val_results = self.validate(epoch)
            if self.scheduler: self.scheduler.step()
            miou = val_results['miou']
            print(f'Epoch {epoch}/{num_epochs} | Loss: {train_loss:.4f}/{val_results["val_loss"]:.4f} | '
                  f'mIoU: {miou:.4f} | Acc: {val_results["pixel_accuracy"]:.4f} | {time.time()-start:.1f}s')
            if miou > self.best_metric:
                self.best_metric = miou
                self.epochs_without_improvement = 0
                self.save_checkpoint(epoch, miou)
                best_results = val_results
                print(f'  * New best mIoU: {miou:.4f}')
            else:
                self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                print(f'\nEarly stopping at epoch {epoch}')
                break
        self.writer.close()
        print(f'\nDone! Best mIoU: {self.best_metric:.4f}')
        return best_results

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[OK] All modules loaded. Device: {device}')
if device == 'cuda': print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# === DATA SETUP ===
# Kaggle: add 2 datasets vao notebook input:
#   1. leehien12/uav-forest-sunny-part1 (seq1-4)
#   2. khesng/uav-forest-sunny-next (seq5-8)

import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

def find_seq_dirs(root):
    """Tim tat ca thu muc seq co chua color/ ben trong, bat ke do sau."""
    found = {}
    for p in sorted(root.rglob('color')):
        seq_dir = p.parent  # thu muc cha cua color/
        if seq_dir.name.startswith('seq'):
            found[seq_dir.name] = seq_dir
    return found

if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    
    for inp in [Path('/kaggle/input/datasets/leehien12/uav-forest-sunny-part1'),
                Path('/kaggle/input/datasets/khesng/uav-forest-sunny-next')]:
        if not inp.exists():
            print(f'[WARNING] Not found: {inp.name}')
            print(f'  -> Add Data > Search: "{inp.name}"')
            continue
        seq_dirs = find_seq_dirs(inp)
        for name, real_path in seq_dirs.items():
            link = DATA_ROOT / name
            if not link.exists():
                os.symlink(real_path, link)
                print(f'[OK] {name} -> {real_path}')
else:
    DATA_ROOT = Path('data/forest_sunny')

# Verify
seqs = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.startswith('seq')])
print(f'\nDATA_ROOT: {DATA_ROOT}')
print(f'Sequences: {seqs} ({len(seqs)} total)')
for s in seqs:
    c = len(list((DATA_ROOT / s / 'color').glob('*.png'))) if (DATA_ROOT / s / 'color').exists() else 0
    print(f'  {s}: {c} images')

OUTPUT_DIR = Path('/kaggle/working/outputs') if IS_KAGGLE else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

In [ ]:
# === INLINE SOURCE CODE ===
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from typing import Optional, Dict, List, Tuple, Callable
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# --- Constants ---
LABEL_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65),
    (130, 76, 0), (152, 251, 152), (151, 126, 171), (250, 150, 0),
    (115, 176, 195), (123, 123, 123), (0, 0, 0),
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]
NUM_CLASSES = len(CLASS_NAMES)

def rgb_to_class_id(label_rgb):
    h, w, _ = label_rgb.shape
    class_map = np.full((h, w), NUM_CLASSES - 1, dtype=np.int64)
    for cid, color in enumerate(LABEL_COLORS):
        mask = np.all(label_rgb == np.array(color, dtype=np.uint8), axis=-1)
        class_map[mask] = cid
    return class_map

def class_id_to_rgb(class_map):
    h, w = class_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, color in enumerate(LABEL_COLORS):
        rgb[class_map == cid] = color
    return rgb

# --- Dataset ---
class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        for seq in sequences:
            color_dir = self.root / seq / 'color'
            label_dir = self.root / seq / 'labels'
            if not color_dir.exists():
                print(f'Warning: {color_dir} not found, skipping.')
                continue
            for cf in sorted(color_dir.glob('*.png')):
                lf = label_dir / cf.name
                if lf.exists():
                    self.samples.append((cf, lf))
        print(f'ForestDataset: {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        cp, lp = self.samples[idx]
        image = np.array(Image.open(cp).convert('RGB'))
        label_rgb = np.array(Image.open(lp).convert('RGB'))
        mask = rgb_to_class_id(label_rgb)
        if self.transform:
            t = self.transform(image=image, mask=mask)
            image, mask = t['image'], t['mask']
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).long()
        return image, mask

# --- Transforms ---
def get_train_transforms(img_size=(512, 512)):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3, border_mode=0),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.1),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=(512, 512)):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# --- Splits ---
SEQUENCE_INFO = {
    'seq1': {'pitch': 0, 'altitude': 30},
    'seq2': {'pitch': -60, 'altitude': 30},
    'seq3': {'pitch': -90, 'altitude': 30},
    'seq4': {'pitch': 0, 'altitude': 50},
    'seq5': {'pitch': -60, 'altitude': 50},
    'seq6': {'pitch': -90, 'altitude': 50},
    'seq7': {'pitch': 0, 'altitude': 80},
    'seq8': {'pitch': -60, 'altitude': 80},
    'seq9': {'pitch': -90, 'altitude': 80},
}

def get_available_sequences(data_root):
    root = Path(data_root)
    return sorted([d.name for d in root.iterdir() if d.is_dir() and d.name.startswith('seq')])

def get_split_for_available(data_root):
    avail = get_available_sequences(data_root)
    print(f'Available sequences: {avail}')
    if len(avail) >= 4:
        return {'train': avail[:-2], 'val': [avail[-2]], 'test': [avail[-1]]}
    elif len(avail) == 3:
        return {'train': avail[:1], 'val': [avail[1]], 'test': [avail[2]]}
    elif len(avail) == 2:
        return {'train': avail[:1], 'val': avail[1:], 'test': avail[1:]}
    else:
        return {'train': avail, 'val': avail, 'test': avail}

def print_split_info(split):
    for subset, seqs in split.items():
        if not seqs: continue
        print(f'\n{subset.upper()}:')
        for seq in seqs:
            info = SEQUENCE_INFO.get(seq, {})
            print(f'  {seq}: pitch={info.get("pitch", "?")} deg, alt={info.get("altitude", "?")}m')

# --- Model ---
class UNet(nn.Module):
    def __init__(self, encoder='resnet34', encoder_weights='imagenet', num_classes=11, activation=None):
        super().__init__()
        self.model = smp.Unet(encoder_name=encoder, encoder_weights=encoder_weights,
                              in_channels=3, classes=num_classes, activation=activation)
        self.encoder_name = encoder
        self.num_classes = num_classes
    def forward(self, x): return self.model(x)

MODEL_REGISTRY = {
    'unet': lambda **kw: UNet(**kw),
    'unetpp': lambda encoder='resnet34', encoder_weights='imagenet', num_classes=11, **kw:
        smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
    'deeplabv3plus': lambda encoder='resnet50', encoder_weights='imagenet', num_classes=11, **kw:
        smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
}

def build_model(name, **kwargs):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown model: {name}. Available: {list(MODEL_REGISTRY.keys())}')
    return MODEL_REGISTRY[name](**kwargs)

# --- Losses ---
class DiceLoss(nn.Module):
    def __init__(self, num_classes=11, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        inter = (probs * targets_oh).sum(dim=dims)
        card = probs.sum(dim=dims) + targets_oh.sum(dim=dims)
        return 1.0 - ((2.0 * inter + self.smooth) / (card + self.smooth)).mean()

class CEDiceLoss(nn.Module):
    def __init__(self, num_classes=11, ce_weight=1.0, dice_weight=0.5, class_weights=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes=num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
    def forward(self, logits, targets):
        return self.ce_weight * self.ce(logits, targets) + self.dice_weight * self.dice(logits, targets)

# --- Metrics ---
class SegmentationMetrics:
    def __init__(self, num_classes, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [str(i) for i in range(num_classes)]
        self.confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)
    @torch.no_grad()
    def update(self, preds, targets):
        p, t = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        valid = (t >= 0) & (t < self.num_classes)
        p, t = p[valid], t[valid]
        cm = np.bincount(t * self.num_classes + p, minlength=self.num_classes**2)
        self.confusion_matrix += cm.reshape(self.num_classes, self.num_classes)
    def compute(self):
        cm = self.confusion_matrix
        inter = np.diag(cm)
        union = cm.sum(1) + cm.sum(0) - inter
        iou = np.where(union > 0, inter / union, 0.0)
        class_total = cm.sum(axis=1)
        class_acc = np.where(class_total > 0, inter / class_total, 0.0)
        pixel_acc = inter.sum() / cm.sum() if cm.sum() > 0 else 0.0
        valid = union > 0
        miou = iou[valid].mean() if valid.any() else 0.0
        result = {'miou': float(miou), 'pixel_accuracy': float(pixel_acc),
                  'mean_class_accuracy': float(class_acc[valid].mean()) if valid.any() else 0.0}
        for i, name in enumerate(self.class_names):
            result[f'iou_{name}'] = float(iou[i])
        return result
    def get_confusion_matrix(self):
        return self.confusion_matrix.copy()

# --- Trainer ---
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, config=None, device='cuda'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config = config or {}
        self.device = device
        self.use_amp = self.config.get('amp', True)
        self.scaler = GradScaler(enabled=self.use_amp)
        self.accumulation_steps = self.config.get('accumulation_steps', 1)
        self.writer = SummaryWriter(log_dir=self.config.get('log_dir', 'outputs/logs'))
        self.checkpoint_dir = Path(self.config.get('checkpoint_dir', 'outputs/checkpoints'))
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.save_top_k = self.config.get('save_top_k', 3)
        self.patience = self.config.get('early_stopping', {}).get('patience', 15)
        self.best_metric = 0.0
        self.epochs_without_improvement = 0
        self.metrics = SegmentationMetrics(self.config.get('num_classes', 11), CLASS_NAMES)
        self.global_step = 0
        self.best_checkpoints = []

    def train_epoch(self, epoch):
        self.model.train()
        total_loss, num_batches = 0.0, 0
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [Train]')
        self.optimizer.zero_grad()
        for batch_idx, (images, masks) in enumerate(pbar):
            images, masks = images.to(self.device), masks.to(self.device)
            with autocast(device_type='cuda', enabled=self.use_amp):
                loss = self.criterion(self.model(images), masks) / self.accumulation_steps
            self.scaler.scale(loss).backward()
            if (batch_idx + 1) % self.accumulation_steps == 0:
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
            total_loss += loss.item() * self.accumulation_steps
            num_batches += 1
            self.global_step += 1
            pbar.set_postfix({'loss': f'{loss.item() * self.accumulation_steps:.4f}'})
        return total_loss / max(num_batches, 1)

    @torch.no_grad()
    def validate(self, epoch):
        self.model.eval()
        self.metrics.reset()
        total_loss, num_batches = 0.0, 0
        for images, masks in tqdm(self.val_loader, desc=f'Epoch {epoch} [Val]'):
            images, masks = images.to(self.device), masks.to(self.device)
            with autocast(device_type='cuda', enabled=self.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
            total_loss += loss.item()
            num_batches += 1
            self.metrics.update(outputs.argmax(dim=1), masks)
        results = self.metrics.compute()
        results['val_loss'] = total_loss / max(num_batches, 1)
        return results

    def save_checkpoint(self, epoch, metric):
        path = self.checkpoint_dir / f'epoch_{epoch:03d}_miou_{metric:.4f}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'metric': metric}, path)
        self.best_checkpoints.append((metric, path))
        self.best_checkpoints.sort(key=lambda x: x[0])
        while len(self.best_checkpoints) > self.save_top_k:
            _, old = self.best_checkpoints.pop(0)
            if old.exists(): old.unlink()

    def fit(self, num_epochs):
        print(f'\nStarting training for {num_epochs} epochs on {self.device}\n')
        best_results = {}
        for epoch in range(1, num_epochs + 1):
            start = time.time()
            train_loss = self.train_epoch(epoch)
            val_results = self.validate(epoch)
            if self.scheduler: self.scheduler.step()
            miou = val_results['miou']
            print(f'Epoch {epoch}/{num_epochs} | Loss: {train_loss:.4f}/{val_results["val_loss"]:.4f} | '
                  f'mIoU: {miou:.4f} | Acc: {val_results["pixel_accuracy"]:.4f} | {time.time()-start:.1f}s')
            if miou > self.best_metric:
                self.best_metric = miou
                self.epochs_without_improvement = 0
                self.save_checkpoint(epoch, miou)
                best_results = val_results
                print(f'  * New best mIoU: {miou:.4f}')
            else:
                self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                print(f'\nEarly stopping at epoch {epoch}')
                break
        self.writer.close()
        print(f'\nDone! Best mIoU: {self.best_metric:.4f}')
        return best_results

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[OK] All modules loaded. Device: {device}')
if device == 'cuda': print(f'GPU: {torch.cuda.get_device_name()}')

# Hardcoded split for all 9 sequences
split = {
    'train': ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8'],
    'val': ['seq6'],
    'test': ['seq9'],
}
avail = get_available_sequences(DATA_ROOTS)
print(f'Available sequences: {avail}')
print_split_info(split)

train_tf = get_train_transforms(img_size=IMG_SIZE)
val_tf = get_val_transforms(img_size=IMG_SIZE)

train_ds = ForestDataset(DATA_ROOTS, split['train'], transform=train_tf)
val_ds = ForestDataset(DATA_ROOTS, split['val'], transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nTrain: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')

In [ ]:
MODEL_NAME = 'unet'
ENCODER = 'resnet34'
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

print(f'Model: {MODEL_NAME} ({ENCODER})')
print(f'Image size: {IMG_SIZE}, Batch: {BATCH_SIZE}, Epochs: {EPOCHS}')

## 2. Data Loading

In [ ]:
split = get_split_for_available(DATA_ROOT)
print_split_info(split)

train_tf = get_train_transforms(img_size=IMG_SIZE)
val_tf = get_val_transforms(img_size=IMG_SIZE)

train_ds = ForestDataset(str(DATA_ROOT), split['train'], transform=train_tf)
val_ds = ForestDataset(str(DATA_ROOT), split['val'], transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nTrain: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')

In [ ]:
# Sanity check
import matplotlib.pyplot as plt

images, masks = next(iter(train_loader))
print(f'Batch: images={images.shape}, masks={masks.shape}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(4, images.shape[0])):
    img = images[i].permute(1,2,0).numpy()
    img = (img * [0.229,0.224,0.225] + [0.485,0.456,0.406]) * 255
    axes[0,i].imshow(img.clip(0,255).astype('uint8'))
    axes[0,i].set_title(f'Image {i}'); axes[0,i].axis('off')
    axes[1,i].imshow(class_id_to_rgb(masks[i].numpy()))
    axes[1,i].set_title(f'Label {i}'); axes[1,i].axis('off')
plt.tight_layout()
plt.show()

## 3. Model & Training

In [ ]:
model = build_model(MODEL_NAME, encoder=ENCODER, num_classes=NUM_CLASSES)
print(f'Model: {MODEL_NAME} ({ENCODER}), Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = CEDiceLoss(num_classes=NUM_CLASSES, ce_weight=1.0, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    criterion=criterion, optimizer=optimizer, scheduler=scheduler,
    config={
        'amp': True, 'accumulation_steps': 1,
        'log_dir': str(OUTPUT_DIR / 'logs'),
        'checkpoint_dir': str(OUTPUT_DIR / 'checkpoints'),
        'save_top_k': 3, 'early_stopping': {'patience': 15},
        'num_classes': NUM_CLASSES,
    },
    device=device,
)

best_results = trainer.fit(num_epochs=EPOCHS)

In [ ]:
print('\nBest Results:')
print(f"  mIoU: {best_results.get('miou', 0):.4f}")
print(f"  Pixel Accuracy: {best_results.get('pixel_accuracy', 0):.4f}")
for name in CLASS_NAMES:
    iou = best_results.get(f'iou_{name}', 0)
    if iou > 0:
        print(f"  IoU {name}: {iou:.4f}")